# Training the Attention U-Net

This notebook covers:
1. Local training on a single site (baseline)
2. Federated training across 3 hospital nodes
3. Evaluating and comparing results

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 1. Local Training (Single Site)

We train on one hospital's data first to make sure the pipeline works.
This also gives us a **baseline** to compare against the federated model.

### 1.1 Create the DataLoaders

In [ ]:
from fedvis.data import FeTSDataset
from torch.utils.data import DataLoader

DATA_ROOT = '../data/FeTS2022'

cfg = OmegaConf.create({
    'data': {
        'processed_path': DATA_ROOT,
        'volume_size': {'depth': 64, 'height': 128, 'width': 128},
        'train_ratio': 0.8,
        'val_ratio': 0.1,
        'test_ratio': 0.1,
        'sites': ['1', '6', '18', '21'],
        'use_modality': 'T1',
        'binary_labels': True,
    },
    'paths': {'data_root': DATA_ROOT},
})

train_ds = FeTSDataset(cfg, split='train', site='1')
val_ds = FeTSDataset(cfg, split='val', site='1')

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples")

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2)

### 1.2 Inspect a Batch

In [ ]:
vol, mask = next(iter(train_loader))
print(f"Volume: {vol.shape}  dtype={vol.dtype}")
print(f"Mask:   {mask.shape}  dtype={mask.dtype}")
print(f"Value range: [{vol.min():.2f}, {vol.max():.2f}]")
print(f"Tumor voxels: {mask.sum().item()} / {mask.numel()}")

# show a middle slice
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
mid = vol.shape[2] // 2
axes[0].imshow(vol[0, 0, mid].numpy(), cmap='gray')
axes[0].set_title('MRI Slice')
axes[1].imshow(mask[0, 0, mid].numpy(), cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Tumor Mask')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

### 1.3 Train the Model

In [ ]:
from fedvis.models.attention_unet import AttentionUNet3D
from fedvis.training.trainer import Trainer

model = AttentionUNet3D(in_channels=1, out_channels=1, base_features=32)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg={'lr': 1e-4, 'epochs': 50, 'dice_weight': 1.0, 'bce_weight': 1.0},
    output_dir='../outputs/local_brain_site1',
    device=device,
)

In [ ]:
# train for 50 epochs
best_dice = trainer.train(num_epochs=50)
print(f"\nBest validation dice: {best_dice:.4f}")

### 1.4 Visualize Training Results

In [ ]:
# load the best model and run inference on a val sample
ckpt = torch.load('../outputs/local_brain_site1/checkpoints/best.pth', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()

vol, mask = val_ds[0]
vol = vol.unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(vol))

pred_np = pred[0, 0].cpu().numpy()
mask_np = mask[0].numpy()
vol_np = vol[0, 0].cpu().numpy()

mid = pred_np.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(vol_np[mid], cmap='gray')
axes[0].set_title('Input')
axes[1].imshow(mask_np[mid], cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Ground Truth')
axes[2].imshow(pred_np[mid], cmap='Reds', vmin=0, vmax=1)
axes[2].set_title(f'Prediction (dice={best_dice:.3f})')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

---

## 2. Federated Training

Now we simulate 3 hospitals collaborating without sharing data.
Each hospital has its own organ-specific dataset:

| Node | Organ | Dataset | Site |
|------|-------|---------|------|
| 0 | Brain | FeTS 2022 | 1 |
| 1 | Prostate | Multi-site Prostate | BIDMC |
| 2 | Lung | CT Lung | 1 |

To run the full simulation from the command line:
```bash
python -m fedvis.scripts.train_federated --rounds 20 --local_epochs 1
```

Below we show the federated training concept inline.

### 2.1 How Federated Averaging Works

```
Round N:
  1. Server sends global model W to all 3 hospitals
  2. Each hospital trains locally for 1 epoch → W_brain, W_prostate, W_lung
  3. Server computes: W_new = Σ (n_k / N) × W_k
     (weighted by each hospital's sample count)
  4. Repeat
```

In [ ]:
# simplified fedavg demo with just the brain dataset split into 3 "nodes"
import copy
from fedvis.models.losses import dice_coefficient

# create 3 models starting from the same weights
global_model = AttentionUNet3D(in_channels=1, out_channels=1, base_features=32)

print("Simulating FedAvg with 3 nodes...")
print("(Using brain data split into 3 partitions for demo)\n")

# helper: weighted average of model parameters
def fedavg(models, weights):
    """Average model parameters weighted by sample counts."""
    avg_state = copy.deepcopy(models[0].state_dict())
    for key in avg_state:
        avg_state[key] = sum(
            w * m.state_dict()[key].float() for m, w in zip(models, weights)
        )
    return avg_state

print("FedAvg function ready.")
print("Run the CLI script for full 3-dataset federation:")
print("  python -m fedvis.scripts.train_federated --rounds 20")

---

## 3. Evaluation

After training, compare:
- **Local model:** Trained on one site only
- **Federated model:** Trained across 3 sites with FedAvg

Expected result: the federated model should generalize better to unseen sites.

In [ ]:
# placeholder for post-training comparison
# fill these in after running both training modes

results = {
    'Local (site 1)': {'dice': 0.0, 'comment': 'run train_local first'},
    'Federated (3 sites)': {'dice': 0.0, 'comment': 'run train_federated first'},
}

for name, r in results.items():
    print(f"{name}: dice={r['dice']:.4f}  ({r['comment']})")

## Next Steps

1. Download datasets (see `01_data_setup.ipynb`)
2. Run local training above or via CLI
3. Run federated training via CLI
4. Fill in the evaluation table and compare results
5. Use the best model for the Flask API and frontend